# Leseführung

Die Arbeit ist über die auf GitHub publizierten HTML-Seiten der einzelnen Skripts dokumentiert. Dieses übergeordnete Skript [PWRG2](https://riedevik.github.io/PWRG2/PWRG2.html) dient als Einstieg und fasst Kontext, Zielsetzung, Daten, Workflow, methodische Entscheidungen, zentrale Resultate sowie Herausforderungen zusammen.

Die drei technischen Skripts dokumentieren die einzelnen Verarbeitungsschritte:

- [Skript 1) Polygon der Area of Interest (AOI) vorbereiten](https://riedevik.github.io/PWRG2/1_AOI_Polygon_vorbereiten.html)
- [Skript 2) STAC-Suche bis Composite-Erstellung](https://riedevik.github.io/PWRG2/2_STAC_Suche_bis_Composites.html)
- [Skript 3) Change Detection & Zeitreihenanalyse](https://riedevik.github.io/PWRG2/3_Change_Detection_und_Zeitreihenanalyse.html)

Zu Beginn jedes Skripts wird der `Zweck des Skripts` sowie der übergeordnete `Workflow` beschrieben. Zusätzlich werden der detailliertere Workflow sowie zentrale Codeabschnitte im Verlauf der Skripts kommentiert, um die methodischen Entscheidungen und Verarbeitungsschritte nachvollziehbar zu machen.

*Technischer Hinweis (nicht relevant für Korrektur): Um die Zellen der Skripts erneut auszuführen ist es wichtig, die Reihenfolge der Skripts einzuhalten, da die Inputs auf Outputs vorheriger Skripts basieren. Die Reihenfolge lautet: 1) / 2) / 3) / PWRG. Das Skript PWRG2 muss als letztes ausgeführt werden, da die Ergebnisdarstellungen auf den vorherigen Skripts basieren.*

# Einleitung und Kontext 

Die rumänischen Karpatenwälder gehören zu den ökologisch wertvollsten Waldökosystemen Europas und umfassen bedeutende Ur- und Primärwaldflächen. Obwohl viele dieser Wälder national und durch europäische Naturschutzrichtlinien geschützt sind, ist Rumänien seit Jahren von Waldverlust betroffen. Als zentrale Ursachen gelten unter anderem illegaler Holzschlag, Misswirtschaft und Schwierigkeiten bei der Durchsetzung von Umweltgesetzen [@ianc2025].

Da lokale Kontrollen teilweise erschwert oder umgangen werden können, bietet die Fernerkundung eine wichtige Ergänzung zur Überwachung von Waldveränderungen. Satellitendaten ermöglichen eine grossflächige, regelmässige und vergleichsweise unabhängige Erfassung potenzieller Abholzungsereignisse [@hirschmugl2017]. Dies ist insbesondere für Schutzgebiete relevant, in denen Waldveränderungen ökologische Funktionen, Lebensräume und Schutzziele beeinträchtigen können. Vor diesem Hintergrund untersucht das Projekt Waldveränderungen in einem rumänischen Natura-2000-Gebiet anhand von Sentinel-2-Daten.

## Area of Interest (AOI)

Als Untersuchungsgebiet wurde das Natura-2000-Gebiet **[Râul Târgului – Argeșel – Râușor (ROSAC0381)](https://biodiversity.europa.eu/sites/natura2000/ROSAC0381)** in Rumänien gewählt. Das Gebiet liegt in den rumänischen Karpaten und umfasst überwiegend bewaldete Flächen, wodurch es sich für eine Analyse von Waldveränderungen auf Basis von Sentinel-2-Daten eignet. Die Fläche beläuft sich auf rund 13'176 Hektar (132 km²).

[Natura 2000](https://natura2000.eea.europa.eu/) ist ein europäisches Schutzgebietsnetzwerk, das dem Erhalt gefährdeter Arten und Lebensräume dient. Veränderungen der Waldflächen innerhalb solcher Schutzgebiete sind daher besonders relevant, da sie potenziell Auswirkungen auf ökologische Funktionen, Lebensräume und Schutzziele haben können.

Für die räumliche Abgrenzung der Analyse wurde das Polygon des Natura-2000-Gebiets verwendet. Dieses wurde aus dem Natura-2000-Datensatz extrahiert, in das Koordinatensystem WGS 84 (EPSG:4326) transformiert und als GeoJSON-Datei gespeichert. Das Polygon dient anschliessend als Grundlage für die STAC-Suche, die Szenenauswahl, das Clipping der Sentinel-2-Daten und alle weiteren räumlichen Analysen.

In [2]:
# Library laden
import geopandas as gpd
import folium

# Polygon der AOI einlesen 
aoi = gpd.read_file("polygon_aoi.geojson")

# CRS auf WGS84 setzen 
aoi = aoi.to_crs("EPSG:4326")

# Geometrie des Polygons ableiten 
aoi_geom = aoi.geometry.iloc[0]

# Visualisieren

centroid = aoi_geom.centroid

m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)

folium.GeoJson(aoi_geom).add_to(m)

m

# Zielsetzung und Fragestellung

Ziel des Projekts ist es, einen reproduzierbaren Workflow zur Erkennung potenzieller Waldverluste in einem rumänischen Natura-2000-Gebiet mit Sentinel-2-Daten aufzubauen. Der Fokus liegt dabei nicht auf einer vollständig validierten Waldverlustkartierung und deren Interpretation, sondern auf dem methodischen Aufbau eines nachvollziehbaren Python-Workflows zur Datenbeschaffung, Vorverarbeitung, Berechnung spektraler Indizes, Change Detection, Zeitreihenanalyse und Visualisierung.

Die übergeordnete Fragestellung lautet:

**Wie lassen sich Sentinel-2-Daten und spektrale Indizes nutzen, um potenzielle Waldverluste in einem ausgewählten rumänischen Natura-2000-Gebiet sichtbar zu machen, und wie kann dafür ein reproduzierbarer Workflow in Python aufgebaut werden?**

Daraus ergeben sich drei Teilziele:

1. Aufbau einer reproduzierbaren Datenpipeline zur Verarbeitung von Sentinel-2-Level-2A-Daten in Python, einschliesslich STAC-Suche, Szenenauswahl, Vorverarbeitung und Indexberechnung.

2. Anwendung einfacher Change-Detection- und Zeitreihenanalyse-Ansätze auf Basis spektraler Indizes, um potenzielle Waldveränderungen zwischen 2021 und 2025 sichtbar zu machen.

3. Entwicklung geeigneter Visualisierungen, um die räumlichen Muster der detektierten Veränderungen nachvollziehbar darzustellen.

# Daten und Untersuchungszeitraum

Für die Analyse wurden **Sentinel-2-Level-2A-Daten** verwendet. Diese enthalten atmosphärisch korrigierte Oberflächenreflexionen und eignen sich daher für die Berechnung spektraler Vegetations- und Störungsindizes. Die Daten wurden über den STAC-Katalog der Microsoft Planetary Computer Platform gesucht und verarbeitet.

Untersucht wurde der Zeitraum **2021 bis 2025**. Für jedes Jahr wurde eine möglichst wolkenarme Sentinel-2-Szene innerhalb des Monats Juli ausgewählt. Die Beschränkung auf einen vergleichbaren Sommerzeitraum reduziert saisonale Unterschiede in der Vegetation und verbessert die Vergleichbarkeit der spektralen Indizes zwischen den Jahren.

Für die Analyse wurden insbesondere folgende Sentinel-2-Bänder verwendet:

* **B02:** Blau
* **B03:** Grün
* **B04:** Rot
* **B08:** Nahinfrarot
* **B12:** Kurzwelliges Infrarot
* **SCL:** Scene Classification Layer zur Maskierung von Wolken, Wolkenschatten und Cirrus

Auf Basis dieser Bänder wurden zwei spektrale Indizes berechnet: der **NDVI** zur Beschreibung der Vegetationsvitalität und der **NBR** zur Erfassung von Veränderungen in Vegetationsstruktur und Feuchtigkeitszustand.

## Wahl der spektralen Indizes

Zur Analyse potenzieller Waldveränderungen wurden der **NDVI** und der **NBR** berechnet. Spektrale Indizes werden in der Fernerkundung häufig eingesetzt, da sie bestimmte Eigenschaften der Vegetation hervorheben und Veränderungen zwischen Zeitpunkten besser sichtbar machen als einzelne Spektralbänder [@zhou2023].

Der NDVI kombiniert das rote und das nahinfrarote Spektralband:

$$
NDVI = \frac{B08 - B04}{B08 + B04}
$$

Er beschreibt die Grünheit bzw. photosynthetische Aktivität der Vegetation und eignet sich daher als allgemeiner Indikator für Vegetationszustand und Vegetationsdichte [@zhou2023].

Der NBR kombiniert das nahinfrarote und das kurzwellige Infrarotband:

$$
NBR = \frac{B08 - B12}{B08 + B12}
$$

Er reagiert besonders sensitiv auf Veränderungen in Vegetationsstruktur, Feuchtigkeit und Störungen der Walddecke. Studien zeigen, dass SWIR-basierte Feuchte- bzw. Störungsindizes wie NBR oder NDMI bei abrupten negativen Vegetationsveränderungen häufig sensitiver reagieren als reine Grünheitsindizes wie NDVI [@zhou2023].

# Methodischer Workflow

Der Workflow wurde in drei technische Skripts aufgeteilt. Dadurch bleiben die einzelnen Verarbeitungsschritte übersichtlich und können unabhängig voneinander nachvollzogen werden.

## Vorbereitung der AOI

Im [Skript 1](https://riedevik.github.io/PWRG2/1_AOI_Polygon_vorbereiten.html) wird die Area of Interest vorbereitet. Dazu wird das Polygon des Natura-2000-Gebiets aus dem Natura-2000-Datensatz extrahiert, in `WGS 84` transformiert und als GeoJSON-Datei gespeichert. Dieses Polygon bildet die räumliche Grundlage für alle folgenden Verarbeitungsschritte.

## STAC-Suche und Szenenauswahl

Im [Skript 2](https://riedevik.github.io/PWRG2/2_STAC_Suche_bis_Composites.html) erfolgt die Suche und Vorverarbeitung der Sentinel-2-Daten. Für jedes Untersuchungsjahr werden geeignete Sentinel-2-Level-2A-Szenen über den STAC-Katalog gesucht. Dabei werden nur Szenen berücksichtigt, welche die AOI ausreichend abdecken und innerhalb des definierten Untersuchungszeitraums liegen.

Da die AOI von mehreren Sentinel-2-Tiles geschnitten werden kann, werden die Suchresultate nach Aufnahmedatum gruppiert. Für jedes Datum wird geprüft, ob die Kombination der verfügbaren Tiles die AOI ausreichend abdeckt. Für die verbleibenden Szenen  wird anschliessend der Anteil von Wolken und Wolkenschatten innerhalb der AOI berechnet, indem der SCL-Layer verwendet wird. Zum Schluss wird für jedes Jahr die Szene mit dem geringsten berechneten Wolkenanteil innerhalb der AOI ausgewählt.

## Vorverarbeitung und Indexberechnung

Die Vorverarbeitung und Indexberechnung wird ebenfalls im [Skript 2](https://riedevik.github.io/PWRG2/2_STAC_Suche_bis_Composites.html) durchgeführt. Nach der Szenenauswahl werden die benötigten Sentinel-2-Bänder geladen und für die weitere Analyse vorbereitet. Verwendet werden die Bänder `B02`, `B03`, `B04`, `B08` und `B12` sowie der Scene Classification Layer (`SCL`). Die Spektralbänder bilden die Grundlage für die RGB-Composites sowie für die Berechnung der spektralen Indizes NDVI und NBR. Der SCL-Layer wird verwendet, um Wolken, Wolkenschatten und Cirrus zu erkennen und aus den Analysen auszuschliessen.

Da die AOI von mehreren Sentinel-2-Tiles abgedeckt wird, werden die Tiles pro Jahr mosaikiert und die Rasterdaten anschliessend auf die AOI zugeschnitten. Um Pixel mit Wolken oder Wolkenschatten von der Analyse auszuschliessen wird anhand des SCL-Layers pro Raster eine Wolkenmaske erstellt, und die betroffenen Pixel auf `NA` gesetzt. Für die Zeitreihenanalyse werden für jedes Jahr Raster (`ValidMask`) mit den Wolkenpixeln erstellt, um diese in einem späteren Schritt zu kombinieren und auf alle Raster anzuwenden, damit eine lückenlose Analyse ermöglicht wird.

Zuletzt werden RGB-, NDVI- und NBR-Composites erstellt. Diese Zwischenergebnisse werden als GeoTIFF-Dateien gespeichert und bilden die Grundlage für die anschliessende Change Detection und Zeitreihenanalyse.

## Preprocessing der Rasterdaten für die Analyse

Im [Skript 3](https://riedevik.github.io/PWRG2/3_Change_Detection_und_Zeitreihenanalyse.html) werden zunächst die in Skript 2 erstellten Rasterdaten geladen. Dazu gehören die jährlichen RGB-, NDVI-, NBR-, sowie die ValidMask-Raster. 

Zunächst werden farbgetreue RGB-Composites erstellt zur Kontrolle und Darstellung der Change Detection. Anschliessend werden die NDVI- und NBR-Raster für die weitere Analyse vorbereitet. Dazu gehört in einem ersten Schritt der Ausschluss von Nicht-Vegetations-Pixeln, welche anhand eines NDVI-Thresholds klassifiziert werden. Die von Nicht-Vegetation bereinigten Raster sind die Basis für die Change Detection. In einem zweiten Schritt werden die ValidMask-Raster mit den Wolkenpixeln der einzelnen Jahre zu einer kombinierten Maske über alle Jahre zusammengefügt. Für die Zeitreihenanalyse werden die Raster zusätzlich von den Wolkenpixeln aller Jahre bereinigt, um eine lückenlose Analyse zu ermöglichen.

## Change Detection und Zeitreihenanalyse

Die **Change Detection** erfolgt auf Basis von NDVI-Differenzen zwischen zwei aufeinanderfolgenden Jahren. Dazu werden jeweils die NDVI-Werte zweier Jahre miteinander verglichen. Für jedes Pixel wird berechnet, wie stark sich der NDVI zwischen dem früheren und dem späteren Jahr verändert hat.

$$
dNDVI = NDVI_{t2} - NDVI_{t1}
$$

Stark negative dNDVI-Werte weisen auf eine deutliche Abnahme der Vegetationsvitalität hin. Solche Veränderungen können auf potenzielle Waldverluste oder andere Störungen der Vegetationsdecke hindeuten. Mithilfe manuell gesetzter Thresholds werden jene Pixel abgegrenzt, bei denen die NDVI-Abnahme stark genug ist, um als potenzielles Abholzungsereignis interpretiert zu werden.

Die detektierten Veränderungspixel werden anschliessend auf den RGB-Composites visualisiert. Ebenfalls wurde die Waldverlustfläche in Hektar berechnet und über die Jahre mittels Barplot dargestellt.

Ergänzend zur paarweisen Change Detection wurde eine einfache **Zeitreihenanalyse** durchgeführt. Dafür wurden die jährlichen NDVI- und NBR-Werte über den Zeitraum 2021 bis 2025 betrachtet. Ziel ist es, nicht nur Veränderungen zwischen zwei aufeinanderfolgenden Jahren sichtbar zu machen, sondern auch die Entwicklung der spektralen Indizes über mehrere Jahre hinweg zu untersuchen.

Einerseits wurden die globalen Mittelwerte pro Jahr, und andererseits ein pixelweiser linearer Trend berechnet. Die Ergebnisse wurden mit Rasterkarten der Steigungswerte, Histogrammen und Boxplot dargestellt.

In der Zeitreihenanalyse werden sowohl NDVI als auch NBR berücksichtigt, da die beiden Indizes unterschiedliche Eigenschaften der Vegetation abbilden. Der Vergleich beider Indizes hilft bei der Interpretation der Ergebnisse. Wenn NDVI und NBR ähnliche negative Entwicklungen zeigen, kann dies auf eine robustere Veränderung der Vegetation hindeuten. 

# 6 Ergebnisse und Interpretation

## 6.1 Szenenauswahl
Kurze Tabelle: Jahr, Datum, Anzahl Tiles, Wolkenanteil AOI.

Hier wird die Tabelle mit dem Ergebnis der STAC-Suche nach der finalen Szenenauswahl präsentiert. Diese Szenen wurden für die weiteren Analysen verwendet. Sie sind die wolkenärmsten innerhalb des Untersuchungszeitraums pro Jahr.

In [3]:
import pandas as pd

best_scenes_df = pd.read_csv("best_scenes_table.csv")
best_scenes_df

,Jahr,Datum,Wolkenanteil_AOI_%,Anzahl_Tiles,AOI_Abdeckung_%
0,2021,2021-07-29,0.0003,2,100.0
1,2022,2022-07-22,0.0000,2,100.0
2,2023,2023-07-09,0.3906,4,100.0
3,2024,2024-07-13,1.3588,2,100.0
4,2025,2025-07-26,0.0000,2,100.0


## 6.2 Change Detection
Kurze Interpretation der dNDVI-Ergebnisse:
- Welche Perioden zeigen die grössten potenziellen Waldverluste?
- Sind die Veränderungspixel räumlich konzentriert oder verteilt?
- Hinweis: Es handelt sich um potenzielle Waldverluste, nicht validierte Abholzung.



## 6.3 Zeitreihenanalyse
Kurze Interpretation:
- Entwicklung der globalen NDVI-/NBR-Mittelwerte.
- Auffälligkeit 2021.
- Vergleich Trend 2021–2025 vs. Sensitivität 2022–2025.
- Wo stimmen NDVI und NBR räumlich überein?



## 6.4 Methodische Einordnung
Die Resultate zeigen, dass der Workflow geeignet ist, potenzielle Waldveränderungen sichtbar zu machen. Eine abschliessende Validierung ist jedoch nicht Teil der Arbeit.

# Herausforderungen:

Eine zentrale Herausforderung bestand darin, den Zugriff auf Sentinel-2-Daten über den STAC-Katalog und die Microsoft Planetary Computer Platform zu verstehen. Besonders relevant war dabei das Prinzip des *lazy loading*. Viele Verarbeitungsschritte werden zunächst nur vorbereitet und erst dann tatsächlich ausgeführt, wenn die Daten explizit geladen oder berechnet werden. Für den Aufbau des Workflows war es daher wichtig zu verstehen, wann Daten nur referenziert werden und wann sie tatsächlich verarbeitet werden.

Eine weitere Herausforderung war die Entwicklung einer geeigneten Logik zur Auswahl der wolkenärmsten Szene pro Untersuchungsjahr. Der in den STAC-Metadaten angegebene Wolkenanteil bezieht sich nicht zwingend auf die AOI und ist daher für diese Analyse nur bedingt aussagekräftig. Deshalb wurde der Wolkenfilter in der STAC-Suche bewusst relativ offen gewählt. Anschliessend wurde für jede geeignete Szene der Wolkenanteil innerhalb der AOI anhand des Scene Classification Layer berechnet. Erst auf dieser Grundlage wurde die beste Szene pro Jahr ausgewählt.

Auch der Umgang mit `stackstac` erforderte Einarbeitung. Das Tool ist sehr nützlich, weil es STAC-Items direkt als xarray-Datenstruktur laden und weiterverarbeiten kann. Gleichzeitig war es notwendig zu verstehen, wie Funktionen auf diese Daten angewendet werden, wie Mosaike aus mehreren Tiles erstellt werden und wann Berechnungen tatsächlich ausgeführt werden.

Ein wichtiger Teil des Skill-Aufbaus bestand zudem darin, eigene Funktionen und Loops nachvollziehbar zu strukturieren. Dabei wurde jeweils zuerst ein einzelnes Testjahr verarbeitet, um die Funktionalität und den Output zu verstehen. Erst danach wurden die Funktionen auf alle Untersuchungsjahre angewendet. Dieses Vorgehen half dabei, Fehler früh zu erkennen und den Workflow schrittweise aufzubauen.

Zusätzlich war es notwendig, den Umgang mit Listen und Dictionaries besser zu verstehen, da viele Zwischenergebnisse nach Jahr oder Szenendatum gespeichert und später wiederverwendet werden mussten. Insbesondere für die Organisation von Best Scenes, RGB-Kompositen, NDVI-/NBR-Rastern, ValidMasks und Change-Detection-Ergebnissen waren Dictionaries hilfreich.

Eine methodische Herausforderung zeigte sich bei den stark unterschiedlichen NDVI-Werten zwischen den Jahren, insbesondere im Jahr 2021. Dadurch liess sich kein einheitlicher Threshold sinnvoll auf alle Jahresvergleiche anwenden. Die Thresholds für potenzielle Waldverluste mussten daher individuell je Jahresvergleich gewählt werden. Die Ursache der auffälligen Unterschiede kann im Rahmen dieser Arbeit nicht abschliessend geklärt werden. Mögliche Einflussfaktoren sind unterschiedliche Aufnahmezeitpunkte, atmosphärische Bedingungen, Restwolken oder Schatten oder phänologische Unterschiede.

# Verwendung von KI

**ChatGPT** wurde verwendet für:

- Verständnisfragen zu Code-Elementen oder Error-Meldungen
- Umschreiben von Text
- Umschreiben / Vereinfachen von Code
- Zum Suchen geeigneten Youtube-Videos zu Python
- Zur Überprüfung der Vollständigkeit anhand des Projektantrags
- Als Hilfe zur Lösung eines Problems; z.B. wie geometrische Duplikate erkannt werden können
- Zum Suchen von Python-Funktionen und Bibliotheken